### Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import itertools
from mogra.datatypes import Shruti, SSwar, SAPTAK_MARKS
from mogra.tonnetz import EFGenus, Tonnetz

### Toy Problem

I have a swarasamooha. I'm operating within a bounded tonnetz net.<br>
I want to evaluate a "cost" of each shruti assignment to every note in the samooha.

e.g.<br>
my samooha is `b1 = Sgn,Sn,SgSggn,n,n,S` and set of notes are `{S, g, n}` with options within my net `{S, g1/g2, n1/n2}`<br>
I'd like to arrive at a parameterization/family `J` that evaluate `J(b1(S,g1,n1)), J(b1(S,g1,n2)), J(b1(S,g2,n1)), J(b1(S,g2,n2))`

Eventually, I'd like to use this for contrastive learning:<br>
for 2 samoohas, `b1, b2` if I know for a fact that `{S, g1, n2}` minimizes `J(b1)` and `{S, g2, n1}` minimizes `J(b2)`,<br>
then I can use this fact to learn `J`

**The nature of J**

0th order
- collapse `b` into a histogram; `J` is not a function of a histogram on notes
- this ignores *movement/momentum*

aribitrary J
- ?

### Setup

In [ ]:
primes = [3, 5]
ef = EFGenus(primes=primes, powers=[3, 1])
tn = Tonnetz(ef)

Harmono-Frequency Space Encoding

In [ ]:
def encode_hf(coordinate, saptak_mark: str):
    """
    TODO: replace the input with a Shruti object
    """
    if saptak_mark not in SAPTAK_MARKS:
        raise ValueError(f"Invalid saptak mark: {saptak_mark}. Must be one of {SAPTAK_MARKS}.")
    freq = float(tn.coord_to_ratio(coordinate))
    return (*coordinate, freq)

Tonnetz encoding

In [ ]:
def create_harmono_frequency_space_paths(samooha: list[SSwar], ground_truth: list[int]):
    bad_paths = []
    good_paths = []
    
    ground_truth_coordinates = np.array(tn.node_coordinates)[ground_truth]
    ground_truth_coordinates = [tuple(row) for row in ground_truth_coordinates]
    
    # take a cartesian product of all options
    all_options = []
    all_saptak_marks = [str(ss)[:-1] for ss in samooha]
    for ss in samooha:
        all_options.append(tn.get_swar_options(ss.swar.name))
    all_paths = itertools.product(*all_options)
    
    for path in all_paths:
        # if all elements of the path are in tn.node_coordinates[ground_truth]:
        if all([cc in ground_truth_coordinates for cc in path]):
            # this is a good path
            good_paths.append([
                encode_hf(coordinate, saptak_mark)
                for coordinate, saptak_mark in zip(path, all_saptak_marks)
            ])
        else:
            # this is a bad path
            bad_paths.append([
                encode_hf(coordinate, saptak_mark)
                for coordinate, saptak_mark in zip(path, all_saptak_marks)
            ])
    
    return {
        "good_paths": good_paths,
        "bad_paths": bad_paths,
    }

### Bgsr v/s Bpls

In [ ]:
# raag ground truths for this tonnetz
bgsr = [1, 4, 5, 7, 8, 10, 13]
bmpls = [7, 10, 12, 13, 15, 16, 19]

Test

In [ ]:
# create "good" and "bad" samples given a samooha: list[SSwar]
# e.g.
b = ",n S g m P".split(" ")
b = [SSwar.from_string(s) for s in b]

In [ ]:
samples = create_harmono_frequency_space_paths(b, bmpls)

assert len(samples["good_paths"]) == 1
assert len(samples["bad_paths"]) == 15

Mukhyangas

In [ ]:
all_samples = {}

In [ ]:
bmpls_phrases = [
    ",n S g m P",
    "n D P",
    "n D P",
    "m g R S",
]

In [ ]:
bgsr_phrases = [
    ",n S m",
    "S g m",
    "g R S",
    ",n D",
    "m D n D",
    "m g R S",
]

In [ ]:
for phrase in bmpls_phrases:
    samples = create_harmono_frequency_space_paths([SSwar.from_string(s) for s in phrase.split(" ")], bmpls)
    all_samples[phrase] = {
        "good_paths": samples["good_paths"],
        "bad_paths": samples["bad_paths"]
    }
for phrase in bgsr_phrases:
    samples = create_harmono_frequency_space_paths([SSwar.from_string(s) for s in phrase.split(" ")], bgsr)
    all_samples[phrase] = {
        "good_paths": samples["good_paths"],
        "bad_paths": samples["bad_paths"]
    }

### Darbari v/s SR Asavari

In [ ]:
# raag ground truths for this tonnetz
savr = []
drbr = []

Mukhyangas

In [ ]:
all_samples = {}

In [ ]:
savr_phrases = [
]

In [ ]:
drbr_phrases = [
]

In [ ]:
for phrase in savr_phrases:
    samples = create_harmono_frequency_space_paths([SSwar.from_string(s) for s in phrase.split(" ")], savr)
    all_samples[phrase] = {
        "good_paths": samples["good_paths"],
        "bad_paths": samples["bad_paths"]
    }
for phrase in drbr_phrases:
    samples = create_harmono_frequency_space_paths([SSwar.from_string(s) for s in phrase.split(" ")], drbr)
    all_samples[phrase] = {
        "good_paths": samples["good_paths"],
        "bad_paths": samples["bad_paths"]
    }

### ChatGPT Attempts

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from sklearn.metrics import roc_auc_score

Path Scorer

In [ ]:
class PathScoringModel(nn.Module):
    """
    A model that transforms sequences of node coordinates into a scalar score
    representing the "goodness" of the path.
    
    The model consists of:
    1. A node encoder that projects raw node coordinates (x, y, z) into
       a higher-dimensional embedding space.
    2. A path encoder that uses a GRU to process sequences of node embeddings
       and produce a fixed-size path embedding.
    3. A scoring head that projects the path embedding to a scalar score.
    """
    
    def __init__(self, node_feat_dim=3, hidden_dim=16, node_emb_dim=16, path_dim=32):
        super().__init__()
        
        # Node encoder: from (x, y, z) → embedding
        self.node_proj = nn.Sequential(
            nn.Linear(node_feat_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, node_emb_dim)
        )  # Learnable

        # Path encoder: GRU over node embeddings
        self.path_encoder = nn.GRU(
            input_size=node_emb_dim,
            hidden_size=path_dim,
            batch_first=True
        )  # Learnable

        # Scoring head: project path embedding to scalar
        self.score_head = nn.Sequential(
            nn.Linear(path_dim, 1)
        )  # Learnable

    def forward_path(self, node_feats_seq, path_lens):
        """
        node_feats_seq: [batch_size, max_path_len, node_feat_dim] raw node coordinates
        path_lens: [batch_size] lengths of each path for packing
        """
        batch_size, T, _ = node_feats_seq.shape
        x = self.node_proj(node_feats_seq)  # [batch_size, max_path_len, node_emb_dim]

        packed = nn.utils.rnn.pack_padded_sequence(x, path_lens.cpu(), batch_first=True, enforce_sorted=False)
        _, h = self.path_encoder(packed)  # h: [1, batch_size, path_dim]
        h = h.squeeze(0)  # [batch_size, path_dim]

        score = self.score_head(h).squeeze(-1)  # [batch_size]
        return score

    def forward(self, path1_feats, path1_lens, path2_feats, path2_lens):
        s1 = self.forward_path(path1_feats, path1_lens)
        s2 = self.forward_path(path2_feats, path2_lens)
        return s1, s2


In [ ]:
# Option 1: For contrastive Learning

class PathPairDataset(Dataset):
    def __init__(self, all_samples_dict):
        self.all_samples_dict = all_samples_dict
        self.num_samples = 0
        for kk, vv in self.all_samples_dict.items():
            self.num_samples += len(vv["good_paths"]) + len(vv["bad_paths"])

    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        phrase = np.random.choice(list(self.all_samples_dict.keys()))
        good_paths = self.all_samples_dict[phrase]["good_paths"]
        bad_paths = self.all_samples_dict[phrase]["bad_paths"]
        return {
            "good_path": good_paths[idx % len(good_paths)],
            "bad_path": bad_paths[idx % len(bad_paths)]
        }

In [ ]:
# # Option 2: For classification

# class PathExamplesDataset(Dataset):
#     def __init__(self, all_samples_dict):
#         self.all_samples_dict = all_samples_dict
#         self.num_samples = 0
#         for kk, vv in self.all_samples_dict.items():
#             self.num_samples += len(vv["good_paths"]) + len(vv["bad_paths"])
#         self.p_positive_sampling = 0.5  # Probability of sampling a positive example

#     def __len__(self):
#         return self.num_samples
    
#     def __getitem__(self, idx):
#         # Randomly sample a good or bad path
#         if np.random.rand() < self.p_positive_sampling:
#             path_type = "good_paths"
#         else:
#             path_type = "bad_paths"
#         phrase = np.random.choice(list(self.all_samples_dict.keys()))
#         paths = self.all_samples_dict[phrase][path_type]
#         path = paths[idx % len(paths)]
#         path_tensor = torch.tensor(path, dtype=torch.float32)
#         label = bool(path_type == "good_paths")
#         return {
#             "path": path_tensor,
#             "label": label
#         } 

In [ ]:
def pad_and_stack(paths):
    lengths = torch.tensor([len(p) for p in paths])
    max_len = lengths.max()
    padded = torch.stack([
        F.pad(torch.Tensor(p), (0, 0, 0, max_len - len(p)))  # pad rows (dim=0)
        for p in paths
    ])
    return padded, lengths

def collate_fn(batch):
    """ Used by the dataloader
        to colalte a batch of good and bad paths
    """
    # 1. get good and bad paths
    
    good_paths = [torch.tensor(item['good_path'], dtype=torch.float32) for item in batch]
    bad_paths = [torch.tensor(item['bad_path'], dtype=torch.float32) for item in batch]

    # 2. pad and stack the paths
    
    good_feats, good_lens = pad_and_stack(good_paths)
    bad_feats, bad_lens = pad_and_stack(bad_paths)
    
    # 3. return a dictionary with good and bad features and lengths
    
    return {
        'good_feats': good_feats,   # [batch_size, max_path_len_good, 3]
        'good_len': good_lens,      # [batch_size]
        'bad_feats': bad_feats,     # [batch_size, max_path_len_bad, 3]
        'bad_len': bad_lens         # [batch_size]
    }

dataloader = DataLoader(
    PathPairDataset(all_samples),
    batch_size=8,
    shuffle=True,
    collate_fn=collate_fn
)

In [ ]:
def contrastive_loss(score1, score2, margin=1.0):
    # we want score1 (good) > score2 (bad)
    return F.relu(score2 - score1 + margin).mean()

In [ ]:
# initialize the model
model = PathScoringModel()

# learning parameters
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
num_epochs = 10


In [ ]:
# Example forward pass
for batch in dataloader:
    scores = model.forward(batch['good_feats'], batch['good_len'], batch['bad_feats'], batch['bad_len'])
    print("Scores shape:", scores[0].shape, scores[1].shape)  # [B]
    break  # Just to see one batch

Training

In [ ]:
# Example training step
for batch in dataloader:
    good_path_feats, good_lens = batch['good_feats'], batch['good_len']
    bad_path_feats, bad_lens = batch['bad_feats'], batch['bad_len']

    score_good, score_bad = model(good_path_feats, good_lens, bad_path_feats, bad_lens)
    loss = contrastive_loss(score_good, score_bad)

    loss.backward()
    optimizer.step()

In [ ]:
# full training loop
for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    for batch in dataloader:
        optimizer.zero_grad()
        
        good_path_feats, good_lens = batch['good_feats'], batch['good_len']
        bad_path_feats, bad_lens = batch['bad_feats'], batch['bad_len']

        score_good, score_bad = model(good_path_feats, good_lens, bad_path_feats, bad_lens)
        loss = contrastive_loss(score_good, score_bad)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(dataloader)}")

Inference

For now, assume that `all_samples_val` contains a validation set of raags

In [ ]:
all_samples_val = all_samples

In [ ]:
# get scores for all paths
good_path_scores = []
bad_path_scores = []
for phrase, samples in all_samples_val.items():
    good_feats, good_lens = pad_and_stack(samples["good_paths"])
    good_path_scores.extend(model.forward_path(good_feats, good_lens).detach().numpy().tolist())
    bad_feats, bad_lens = pad_and_stack(samples["bad_paths"])
    bad_path_scores.extend(model.forward_path(bad_feats, bad_lens).detach().numpy())

In [ ]:
plt.scatter(good_path_scores, np.ones(len(good_path_scores)), label='Good Path Scores', marker='o')
plt.scatter(bad_path_scores, np.zeros(len(bad_path_scores)), label='Bad Path Scores', marker='x')
plt.xlabel('Predicted Path Score')
plt.ylabel('Ground truth Path Score')
plt.ylim(-1, 2)
plt.yticks([0, 1])

In [ ]:
# measure cross entropy loss using good_path_scores, bad_path_scores
# good_path_scores have label 1, bad_path_scores have label 0
good_labels = torch.ones(len(good_path_scores))
bad_labels = torch.zeros(len(bad_path_scores))
all_scores = torch.tensor(good_path_scores + bad_path_scores, dtype=torch.float32)
all_labels = torch.cat([good_labels, bad_labels])
print(f"Classification AUC: {roc_auc_score(all_labels.numpy(), all_scores.numpy())}")

